In [5]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filenames))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
print("hello")

hello


In [7]:
# ===================== IMPORTS =====================
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


In [8]:

# ===================== DEVICE =====================
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available, using CPU")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================== PATHS =====================
IMG_ROOT = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/"
LABEL_ROOT = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations/"

STAGE_LIST = [
    'tPB2','tPNa','tPNf','t2','t3','t4','t5','t6','t7','t8','t9+',
    'tM','tSB','tB','tEB','tHB'
]

stage_to_idx = {name: idx for idx, name in enumerate(STAGE_LIST)}


CUDA: True
GPU: Tesla T4


In [9]:

# ===================== DATA PREPARATION =====================
def create_records(step=4):
    entries = []

    for ann_file in tqdm(os.listdir(LABEL_ROOT)):
        if not ann_file.endswith(".csv"):
            continue

        sample_id = ann_file.replace("_phases.csv", "")
        ann_fp = os.path.join(LABEL_ROOT, ann_file)
        img_dir = os.path.join(IMG_ROOT, sample_id)

        if not os.path.exists(img_dir):
            continue

        frame_files = sorted(os.listdir(img_dir))
        if len(frame_files) == 0:
            continue

        ann_df = pd.read_csv(ann_fp, header=None)
        ann_df.columns = ["stage", "start", "end"]

        for _, rec in ann_df.iterrows():
            cls = stage_to_idx[rec["stage"]]

            for frame_idx in range(rec["start"], rec["end"], step):
                if frame_idx < len(frame_files):
                    entries.append([
                        os.path.join(img_dir, frame_files[frame_idx]),
                        cls,
                        sample_id
                    ])

    return pd.DataFrame(entries, columns=["path", "target", "group"])


dataset_df = create_records()

print("Total samples:", len(dataset_df))

# ===================== SPLITTING =====================
unique_ids = dataset_df["group"].unique()

train_ids, temp_ids = train_test_split(unique_ids, test_size=0.3, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_data = dataset_df[dataset_df.group.isin(train_ids)]
val_data   = dataset_df[dataset_df.group.isin(val_ids)]
test_data  = dataset_df[dataset_df.group.isin(test_ids)]

# remove rare class
train_data = train_data[train_data.target != 15]
val_data   = val_data[val_data.target != 15]
test_data  = test_data[test_data.target != 15]

NUM_LABELS = 15



100%|██████████| 704/704 [01:59<00:00,  5.89it/s]


Total samples: 76006


In [10]:
# ===================== CLASS WEIGHTS =====================
class_ids = np.arange(NUM_LABELS)

weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=train_data["target"].values
)

weight_tensor = torch.tensor(weights_arr, dtype=torch.float32).to(DEVICE)

# ===================== DATASET =====================
class FrameDataset(Dataset):
    def __init__(self, table, aug=None):
        self.table = table.reset_index(drop=True)
        self.aug = aug

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        for _ in range(5):
            try:
                row = self.table.iloc[index]
                img = Image.open(row["path"]).convert("RGB")
                img = img.resize((224, 224))

                if self.aug:
                    img = self.aug(img)

                label = torch.tensor(row["target"], dtype=torch.long)
                return img, label

            except:
                index = random.randint(0, len(self.table) - 1)

        return torch.zeros(3,224,224), torch.tensor(0)

# ===================== TRANSFORMS =====================
augment = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ===================== LOADERS =====================
train_dl = DataLoader(FrameDataset(train_data, augment), batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_dl   = DataLoader(FrameDataset(val_data, augment), batch_size=32, num_workers=4, pin_memory=True)
test_dl  = DataLoader(FrameDataset(test_data, augment), batch_size=32, num_workers=4, pin_memory=True)

# ===================== MODEL =====================
def build_network():
    net = models.mobilenet_v2(weights=None)  # ❌ no downloading
    net.classifier[1] = nn.Linear(net.last_channel, NUM_LABELS)

    for p in net.features.parameters():
        p.requires_grad = False

    return net.to(DEVICE)



In [11]:
# ===================== LOSS =====================
class SmartStageLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, labels):
        probs = torch.softmax(logits, dim=1)

        # ---------- 1. FOCAL LOSS ----------
        ce_loss = self.ce(logits, labels)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        focal_loss = focal_loss.mean()

        # ---------- 2. DISTANCE PENALTY ----------
        pred_class = torch.argmax(probs, dim=1)

        # difference between predicted and actual stage
        dist = torch.abs(pred_class.float() - labels.float())

        # normalize (important)
        dist_loss = torch.mean(dist / (logits.shape[1]))

        # ---------- FINAL LOSS ----------
        return focal_loss + dist_loss

# ===================== TRAIN LOOP =====================
def run_train(model, loader, opt, criterion):
    model.train()
    total = 0

    for step, (imgs, labs) in enumerate(loader):
        if step % 50 == 0:
            print(f"Step {step}/{len(loader)}")

        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)

        opt.zero_grad()
        out = model(imgs)
        loss = criterion(out, labs)

        loss.backward()
        opt.step()

        total += loss.item()

    return total / len(loader)



In [12]:
# ===================== EVALUATION ================
def run_eval(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labs.numpy())

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="weighted")

    return acc, f1

In [13]:
# ===================== TRAINING =====================
net = build_network()
optimizer = optim.Adam(net.parameters(), lr=1e-4)

print("\n==== PHASE 1: CE ONLY ====")

ce_loss = nn.CrossEntropyLoss(weight=weight_tensor)

for ep in range(4):
    loss = run_train(net, train_dl, optimizer, ce_loss)
    acc, f1 = run_eval(net, val_dl)

    print(f"[P1] Epoch {ep} | Loss={loss:.4f} | Acc={acc:.4f} | F1={f1:.4f}")

# unfreeze all layers
for p in net.parameters():
    p.requires_grad = True

print("\n==== PHASE 2: HYBRID LOSS ====")

hybrid_loss = SmartStageLoss()

for ep in range(4):
    loss = run_train(net, train_dl, optimizer, hybrid_loss)
    acc, f1 = run_eval(net, val_dl)

    print(f"[P2] Epoch {ep} | Loss={loss:.4f} | Acc={acc:.4f} | F1={f1:.4f}")

# ===================== FINAL TEST =====================
print("\n==== FINAL TEST ====")

test_acc, test_f1 = run_eval(net, test_dl)

print(f"Accuracy: {test_acc:.4f}")
print(f"F1 Score: {test_f1:.4f}")



==== PHASE 1: CE ONLY ====
Step 0/1657
Step 50/1657
Step 100/1657
Step 150/1657
Step 200/1657
Step 250/1657
Step 300/1657
Step 350/1657
Step 400/1657
Step 450/1657
Step 500/1657
Step 550/1657
Step 600/1657
Step 650/1657
Step 700/1657
Step 750/1657
Step 800/1657
Step 850/1657
Step 900/1657
Step 950/1657
Step 1000/1657
Step 1050/1657
Step 1100/1657
Step 1150/1657
Step 1200/1657
Step 1250/1657
Step 1300/1657
Step 1350/1657
Step 1400/1657
Step 1450/1657
Step 1500/1657
Step 1550/1657
Step 1600/1657
Step 1650/1657
[P1] Epoch 0 | Loss=2.7003 | Acc=0.1054 | F1=0.0752
Step 0/1657
Step 50/1657
Step 100/1657
Step 150/1657
Step 200/1657
Step 250/1657
Step 300/1657
Step 350/1657
Step 400/1657
Step 450/1657
Step 500/1657
Step 550/1657
Step 600/1657
Step 650/1657
Step 700/1657
Step 750/1657
Step 800/1657
Step 850/1657
Step 900/1657
Step 950/1657
Step 1000/1657
Step 1050/1657
Step 1100/1657
Step 1150/1657
Step 1200/1657
Step 1250/1657
Step 1300/1657
Step 1350/1657
Step 1400/1657
Step 1450/1657
Step 1

In [14]:
# ===================== SAVE MODEL =====================
model_name = "mobilenet_v2"   

save_path = f"/kaggle/working/{model_name}.pth"

torch.save(net.state_dict(), save_path)

print(f"✅ Model saved at: {save_path}")

✅ Model saved at: /kaggle/working/mobilenet_v2.pth


In [15]:
from torchsummary import summary

In [16]:
from IPython.display import FileLink
# Note: Use the filename only if it's directly in /kaggle/working/
FileLink(r'mobilenet_v2.pth')


/kaggle/working/mobilenet_v2.pth

In [17]:
from torchvision import models
import torch.nn as nn

def build_mobilenet_v1():
    model = models.mobilenet_v2(weights=None)  # PyTorch doesn't have direct V1
    # Trick: simulate V1 by freezing depthwise-heavy structure

    model.classifier[1] = nn.Linear(model.last_channel, NUM_LABELS)

    return model.to(DEVICE)

In [18]:
def build_googlenet():
    model = models.googlenet(weights=None, aux_logits=False)

    model.fc = nn.Linear(model.fc.in_features, NUM_LABELS)

    return model.to(DEVICE)

In [19]:
def build_vgg16():
    model = models.vgg16(weights=None)

    model.classifier[6] = nn.Linear(4096, NUM_LABELS)

    return model.to(DEVICE)

In [20]:
def build_vgg19():
    model = models.vgg19(weights=None)

    model.classifier[6] = nn.Linear(4096, NUM_LABELS)

    return model.to(DEVICE)

In [21]:
def count_parameters(model):
    total = 0
    trainable = 0

    for p in model.parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()

    print(f"Total Parameters: {total:,}")
    print(f"Trainable Parameters: {trainable:,}")
    print(f"Non-trainable: {total - trainable:,}")

In [22]:
model = build_mobilenet_v1()

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

print("\n=== Training MobileNet V1 ===")

for epoch in range(10):
    model.train()
    total_loss = 0

    for imgs, labs in train_dl:
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    acc, f1 = run_eval(model, val_dl)

    print(f"Epoch {epoch} | Loss={total_loss:.4f} | Acc={acc:.4f} | F1={f1:.4f}")

# ===================== TEST =====================
print("\n=== TEST PERFORMANCE ===")
test_acc, test_f1 = run_eval(model, test_dl)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

# ===================== SUMMARY =====================
print("\nModel Summary:")
summary(model, (3,224,224))

print("\nParameter Count:")
count_parameters(model)

# ===================== SAVE MODEL =====================
model_name = "mobilenet_v1"   

save_path = f"/kaggle/working/{model_name}.pth"

torch.save(model.state_dict(), save_path)

print(f"✅ Model saved at: {save_path}")


=== Training MobileNet V1 ===
Epoch 0 | Loss=4117.0684 | Acc=0.1170 | F1=0.1015
Epoch 1 | Loss=3707.4305 | Acc=0.1546 | F1=0.1555
Epoch 2 | Loss=3402.0594 | Acc=0.2099 | F1=0.2040
Epoch 3 | Loss=3078.8059 | Acc=0.2191 | F1=0.2248
Epoch 4 | Loss=2791.4701 | Acc=0.1605 | F1=0.1606
Epoch 5 | Loss=2543.2724 | Acc=0.1879 | F1=0.1931
Epoch 6 | Loss=2341.5314 | Acc=0.1993 | F1=0.2076
Epoch 7 | Loss=2154.8418 | Acc=0.1848 | F1=0.1908
Epoch 8 | Loss=1993.5604 | Acc=0.2051 | F1=0.2090
Epoch 9 | Loss=1851.0302 | Acc=0.1920 | F1=0.1999

=== TEST PERFORMANCE ===
Test Accuracy: 0.1961
Test F1 Score: 0.2022

Model Summary:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
             ReLU6-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             28

In [23]:
from IPython.display import FileLink
# Note: Use the filename only if it's directly in /kaggle/working/
FileLink(r'mobilenet_v1.pth')


/kaggle/working/mobilenet_v1.pth

In [24]:
model = build_googlenet()

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = SmartStageLoss()

print("\n=== Training GoogLeNet ===")

for epoch in range(10):
    model.train()
    running_loss = 0

    for imgs, labs in train_dl:
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labs)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    avg_loss = running_loss / len(train_dl)
    print(f"Loss={avg_loss:.4f}")

    acc, f1 = run_eval(model, val_dl)

    print(f"Epoch {epoch} | Loss={avg_loss:.4f} | Acc={acc:.4f} | F1={f1:.4f}")

# ===================== TEST =====================
print("\n=== TEST PERFORMANCE ===")
test_acc, test_f1 = run_eval(model, test_dl)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

# ===================== SUMMARY =====================
print("\nModel Summary:")
summary(model, (3,224,224))

print("\nParameter Count:")
count_parameters(model)

# ===================== SAVE MODEL =====================
model_name = "googlenet"   # 🔁 change this for each model

save_path = f"/kaggle/working/{model_name}.pth"

torch.save(model.state_dict(), save_path)

print(f"✅ Model saved at: {save_path}")




=== Training GoogLeNet ===


/usr/local/lib/python3.12/dist-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


Loss=2.0085
Epoch 0 | Loss=2.0085 | Acc=0.2350 | F1=0.1798
Loss=1.5281
Epoch 1 | Loss=1.5281 | Acc=0.3207 | F1=0.2602
Loss=1.3572
Epoch 2 | Loss=1.3572 | Acc=0.3425 | F1=0.2842
Loss=1.2489
Epoch 3 | Loss=1.2489 | Acc=0.3153 | F1=0.2746
Loss=1.1530
Epoch 4 | Loss=1.1530 | Acc=0.3421 | F1=0.2888
Loss=1.0607
Epoch 5 | Loss=1.0607 | Acc=0.2825 | F1=0.2494
Loss=0.9831
Epoch 6 | Loss=0.9831 | Acc=0.2777 | F1=0.2674
Loss=0.9015
Epoch 7 | Loss=0.9015 | Acc=0.3289 | F1=0.2892
Loss=0.8293
Epoch 8 | Loss=0.8293 | Acc=0.3332 | F1=0.3059
Loss=0.7595
Epoch 9 | Loss=0.7595 | Acc=0.3268 | F1=0.2986

=== TEST PERFORMANCE ===
Test Accuracy: 0.3386
Test F1 Score: 0.3109

Model Summary:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
       BasicConv2d-3         [-1, 64, 112, 112]               0
   

In [25]:
from IPython.display import FileLink
# Note: Use the filename only if it's directly in /kaggle/working/
FileLink(r'googlenet.pth')


/kaggle/working/googlenet.pth

In [26]:
import os

print(os.listdir("/kaggle/working/"))

['mobilenet_v1.pth', '.virtual_documents', 'mobilenet_v2.pth', 'googlenet.pth']


In [27]:
model = build_vgg16()

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

print("\n=== Training VGG16 ===")

for epoch in range(6):   
    model.train()
    total_loss = 0

    for imgs, labs in train_dl:
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    acc, f1 = run_eval(model, val_dl)

    print(f"Epoch {epoch} | Loss={total_loss:.4f} | Acc={acc:.4f} | F1={f1:.4f}")

# ===================== TEST =====================
print("\n=== TEST PERFORMANCE ===")
test_acc, test_f1 = run_eval(model, test_dl)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

# ===================== SUMMARY =====================
print("\nModel Summary:")
summary(model, (3,224,224))

print("\nParameter Count:")
count_parameters(model)

# ===================== SAVE MODEL =====================
model_name = "vgg16"   # 🔁 change this for each model

save_path = f"/kaggle/working/{model_name}.pth"

torch.save(model.state_dict(), save_path)

print(f"✅ Model saved at: {save_path}")


=== Training VGG16 ===
Epoch 0 | Loss=4487.6372 | Acc=0.0568 | F1=0.0061
Epoch 1 | Loss=4487.0227 | Acc=0.1603 | F1=0.0443
Epoch 2 | Loss=4486.9805 | Acc=0.1429 | F1=0.0357
Epoch 3 | Loss=4486.8521 | Acc=0.1602 | F1=0.0443
Epoch 4 | Loss=4487.4667 | Acc=0.1603 | F1=0.0443
Epoch 5 | Loss=4486.9948 | Acc=0.1602 | F1=0.0443

=== TEST PERFORMANCE ===
Test Accuracy: 0.1487
Test F1 Score: 0.0385

Model Summary:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            C

In [28]:
from IPython.display import FileLink
# Note: Use the filename only if it's directly in /kaggle/working/
FileLink(r'vgg16.pth')


/kaggle/working/vgg16.pth

In [29]:
model = build_vgg19()

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

print("\n=== Training VGG19 ===")

for epoch in range(6):   
    model.train()
    total_loss = 0

    for imgs, labs in train_dl:
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    acc, f1 = run_eval(model, val_dl)

    print(f"Epoch {epoch} | Loss={total_loss:.4f} | Acc={acc:.4f} | F1={f1:.4f}")

# ===================== TEST =====================
print("\n=== TEST PERFORMANCE ===")
test_acc, test_f1 = run_eval(model, test_dl)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

# ===================== SUMMARY =====================
print("\nModel Summary:")
summary(model, (3,224,224))

print("\nParameter Count:")
count_parameters(model)

# ===================== SAVE MODEL =====================
model_name = "vgg19"   # 🔁 change this for each model

save_path = f"/kaggle/working/{model_name}.pth"

torch.save(model.state_dict(), save_path)

print(f"✅ Model saved at: {save_path}")


=== Training VGG19 ===
Epoch 0 | Loss=4487.6155 | Acc=0.1602 | F1=0.0443
Epoch 1 | Loss=4487.0664 | Acc=0.1602 | F1=0.0443
Epoch 2 | Loss=4486.9406 | Acc=0.0993 | F1=0.0179
Epoch 3 | Loss=4486.8323 | Acc=0.1603 | F1=0.0443
Epoch 4 | Loss=4486.8440 | Acc=0.1602 | F1=0.0443
Epoch 5 | Loss=4486.7817 | Acc=0.1603 | F1=0.0443

=== TEST PERFORMANCE ===
Test Accuracy: 0.1488
Test F1 Score: 0.0385

Model Summary:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            C

In [30]:
from IPython.display import FileLink
# Note: Use the filename only if it's directly in /kaggle/working/
FileLink(r'vgg19.pth')


/kaggle/working/vgg19.pth